In [53]:
import pandas as pd
import numpy as np
import sklearn

1. Загрузите данные из файла data-logistic.csv. Это двумерная выборка, целевая переменная на которой принимает значения -1 или 1.

In [54]:
data = pd.read_csv('data-logistic.csv', header = None)
x = data[[1,2]].to_numpy()
y = data[0].to_numpy()

2. Убедитесь, что выше выписаны правильные формулы для градиентного спуска. Обратите внимание, что мы используем полноценный градиентный спуск, а не его стохастический вариант!


3. Реализуйте градиентный спуск для обычной и L2-регуляризованной
(с коэффициентом регуляризации 10) логистической регрессии. Используйте длину шага k=0.1. В качестве начального приближения
используйте вектор (0, 0).


In [55]:
def log_reg(x, y, C ,k = 0.1, eps = 1e-6, w1 = 0, w2 = 0):
    w1, w2 = w1, w2
    l = len(y)
    for i in range(10000):
        ex = y * (w1 * x[:, 0] + w2 * x[:, 1])
        sl = 1 - (1/(1 + np.exp(-ex)))
        w1_n = w1 + k *(1/l) * np.sum(y* x[:,0]* sl) - k * C * w1
        w2_n = w2 + k *(1/l) * np.sum(y* x[:,1]* sl) - k * C * w2
        evkl =np.sqrt((w1_n - w1)**2 + (w2_n - w2)**2)
        w1 = w1_n
        w2 = w2_n
        if evkl < eps:
            break
    return w1, w2, i


4. Запустите градиентный спуск и доведите до сходимости (евклидово
расстояние между векторами весов на соседних итерациях должно быть не больше 1e-5). Рекомендуется ограничить сверху число
итераций десятью тысячами.

In [56]:
w1_0, w2_0, _ = log_reg(x, y, C=0)

5. Какое значение принимает AUC-ROC на обучении без регуляризации и при ее использовании? Эти величины будут ответом на
задание. В качестве ответа приведите два числа через пробел. Обратите внимание, что на вход функции roc_auc_score нужно подавать оценки вероятностей, подсчитанные обученным алгоритмом.
Для этого воспользуйтесь сигмоидной функцией: a(x) = 1/(1 +
exp(−w1x1 − w2x2)).

In [57]:
def sigmoid(x):
    return 1/(1 + np.exp(-x))

x1 = x[:,0]
x2 = x[:,1]
auc = sklearn.metrics.roc_auc_score(y, sigmoid(w1_0 * x1 + w2_0 * x2))

w1_reg, w2_reg, _ = log_reg(x, y, C=10)
auc_reg = sklearn.metrics.roc_auc_score(y, sigmoid(w1_reg * x1 + w2_reg * x2))
print(auc, auc_reg)

0.9267619047619047 0.9362857142857142


6. Попробуйте поменять длину шага. Будет ли сходиться алгоритм,
если делать более длинные шаги? Как меняется число итераций
при уменьшении длины шага?


In [58]:
for k in [0.001, 0.01, 0.05, 0.1, 0.3, 0.6, 1,3, 5, 7, 10]:
    w1, w2, i = log_reg(x, y, C=0, k=k)
    auc = sklearn.metrics.roc_auc_score(y, sigmoid(w1 * x1 + w2 * x2))
    print(f"k = {k}, auc = {auc}, i = {i}")
print("Алгоритм не будет сходиться при увеличении шага")
print("Число итераций будет увеличиваться при уменьшении шага")

k = 0.001, auc = 0.9282857142857142, i = 9999
k = 0.01, auc = 0.9268571428571428, i = 2448
k = 0.05, auc = 0.9267619047619047, i = 623
k = 0.1, auc = 0.9267619047619047, i = 339
k = 0.3, auc = 0.9267619047619047, i = 125
k = 0.6, auc = 0.9267619047619047, i = 65
k = 1, auc = 0.9267619047619047, i = 39
k = 3, auc = 0.9364761904761904, i = 9999
k = 5, auc = 0.07323809523809523, i = 9999
k = 7, auc = 0.9358095238095238, i = 9999
k = 10, auc = 0.9366666666666666, i = 9999
Алгоритм не будет сходиться при увеличении шага
Число итераций будет увеличиваться при уменьшении шага


7. Попробуйте менять начальное приближение. Влияет ли оно на чтонибудь?


In [59]:
for w1_n, w2_n in [(0,0), (1, 1), (2, 2), (-1, -1), (-0.5, -0.5)]:
    w1, w2, i = log_reg(x, y, C=0, w1 = w1_n, w2 = w2_n)
    auc =  sklearn.metrics.roc_auc_score(y, sigmoid(w1 * x1 + w2 * x2))
    print(f"w1_n = {w1_n}, w2_n = {w2_n}, w1 = {w1}, w2 = {w2}, auc = {auc}, i = {i}")
print("Влияет на количество итераций, на сходимость алгоритма не влияет")

w1_n = 0, w2_n = 0, w1 = 0.288078515028862, w2 = 0.09173653761803442, auc = 0.9267619047619047, i = 339
w1_n = 1, w2_n = 1, w1 = 0.28807812821392775, w2 = 0.09173689521271462, auc = 0.9267619047619047, i = 325
w1_n = 2, w2_n = 2, w1 = 0.2881383699941085, w2 = 0.09168120572897223, auc = 0.9267619047619047, i = 282
w1_n = -1, w2_n = -1, w1 = 0.28807859572251215, w2 = 0.09173646302005246, auc = 0.9267619047619047, i = 348
w1_n = -0.5, w2_n = -0.5, w1 = 0.28807797229993737, w2 = 0.09173703934889567, auc = 0.9267619047619047, i = 343
Влияет на количество итераций, на сходимость алгоритма не влияет
